In [19]:
# Cell 1. Imports
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import transforms

from captum.attr import Occlusion
import json

In [20]:
# Cell 2. Config
IMAGE_PATH = Path('/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/cases/sweep/bird_111_symmetrical_2_1/gt.png')
ROOT = Path("..").resolve()

# Replace these with your actual models.
# The notebook assumes each model takes a tensor of shape [1, 3, H, W]
# and returns logits of shape [1, n_classes].
SHAPE_MODEL_LABEL = 'shape_biased'
TEXTURE_MODEL_LABEL = 'texture_biased'

OCCLUDER_SIZES = [8, 16, 32, 64]
STRIDE_FRACTION = 0.5
BASELINE_VALUE = 0.0
CLASS_NAMES_JSONL = ROOT / "data" / "class_names.jsonl"


# Use the ImageNet normalization only if your models expect it.
USE_IMAGENET_NORMALIZATION = True
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

with CLASS_NAMES_JSONL.open("r", encoding="utf-8") as f:
    classes = [json.loads(line)["class_name"] for line in f if line.strip()]

print("Loaded class list:", len(classes))
print("First 10 classes:", classes[:10])

Loaded class list: 54
First 10 classes: ['ant', 'bat', 'bear', 'bee', 'beetle', 'bird', 'bug', 'bull', 'butterfly', 'camel']


In [6]:
# Cell 3. Load ONNX models

import onnxruntime as ort


class ONNXWrapper(torch.nn.Module):
    def __init__(self, onnx_path, device='cpu'):
        super().__init__()
        providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device == 'cuda' else ['CPUExecutionProvider']
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)

        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name

    def forward(self, x):
        # x: torch tensor [1, C, H, W]
        x_np = x.detach().cpu().numpy()

        outputs = self.session.run(
            [self.output_name],
            {self.input_name: x_np}
        )[0]

        return torch.from_numpy(outputs).to(x.device)


# === paths (replace with your actual files) ===
SHAPE_ONNX_PATH = Path("/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/models/resnet50_geirhos_tl.onnx")
TEXTURE_ONNX_PATH = Path("/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/models/resnet50_tl_20250829.onnx")


shape_model = ONNXWrapper(SHAPE_ONNX_PATH, device=DEVICE.type).to(DEVICE)
texture_model = ONNXWrapper(TEXTURE_ONNX_PATH, device=DEVICE.type).to(DEVICE)

shape_model.eval()
texture_model.eval()

ONNXWrapper()

In [14]:
# Cell 4. Image loading and preprocessing
pil_img = Image.open(IMAGE_PATH).convert('RGB')
img_np = np.asarray(pil_img)

transform_steps = [
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
]

if USE_IMAGENET_NORMALIZATION:
    transform_steps.append(
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    )

preprocess = transforms.Compose(transform_steps)

input_tensor = preprocess(pil_img).unsqueeze(0).to(DEVICE)

print('Image path:', IMAGE_PATH)
print('Original image shape:', img_np.shape)
print('Tensor shape:', tuple(input_tensor.shape))

Image path: /home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/data/cases/sweep/bird_111_symmetrical_2_1/gt.png
Original image shape: (600, 900, 3)
Tensor shape: (1, 3, 224, 224)


In [15]:
# Cell 5. Utility helpers
def get_logits_and_prediction(model, x, classes=None, topk=5):
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)

        pred_idx = int(torch.argmax(logits, dim=1).item())
        pred_prob = float(probs[0, pred_idx].item())

        result = {
            "pred_idx": pred_idx,
            "pred_prob": pred_prob,
            "logits": logits,
            "probs": probs,
        }

        if classes is not None:
            result["pred_class"] = classes[pred_idx]

            topk_probs, topk_idx = torch.topk(probs, k=topk, dim=1)
            topk_idx = topk_idx[0].cpu().numpy()
            topk_probs = topk_probs[0].cpu().numpy()

            result["topk"] = [
                {
                    "idx": int(i),
                    "class": classes[i],
                    "prob": float(p),
                }
                for i, p in zip(topk_idx, topk_probs)
            ]

    return result


def compute_stride(size, fraction=0.5):
    return max(1, int(round(size * fraction)))


def summarize_attribution_map(attr_map):
    attr_np = attr_map.squeeze(0).detach().cpu().numpy()
    if attr_np.ndim == 3:
        attr_2d = attr_np.mean(axis=0)
    else:
        attr_2d = attr_np
    return attr_2d


def normalize_for_display(x):
    x = np.asarray(x, dtype=float)
    x_min = np.nanmin(x)
    x_max = np.nanmax(x)
    if np.isclose(x_min, x_max):
        return np.zeros_like(x)
    return (x - x_min) / (x_max - x_min)

In [21]:
# Cell 6. Baseline predictions
baseline_results = {}

for label, model in models.items():
    model.eval().to(DEVICE)

    res = get_logits_and_prediction(model, input_tensor, classes=classes, topk=5)

    baseline_results[label] = res

    print(f"\n=== {label} ===")
    print("Prediction:", res["pred_class"], f"(idx={res['pred_idx']})")
    print("Confidence:", round(res["pred_prob"], 4))

    print("Top-5:")
    for item in res["topk"]:
        print(f"  {item['class']} ({item['prob']:.4f})")


=== shape_biased ===
Prediction: bird (idx=5)
Confidence: 0.7966
Top-5:
  bird (0.7966)
  hawk (0.1056)
  eagle (0.0859)
  goose (0.0036)
  owl (0.0022)

=== texture_biased ===
Prediction: dog (idx=17)
Confidence: 1.0
Top-5:
  dog (1.0000)
  shark (0.0000)
  butterfly (0.0000)
  fish (0.0000)
  cat (0.0000)


In [22]:
# Cell 7. Run Captum occlusion sweep
results = []
occlusion_maps = {}

for model_label, model in models.items():
    target_idx = baseline_results[model_label]['pred_idx']
    occlusion = Occlusion(model)
    occlusion_maps[model_label] = {}

    for size in OCCLUDER_SIZES:
        stride = compute_stride(size, STRIDE_FRACTION)

        attr = occlusion.attribute(
            input_tensor,
            target=target_idx,
            sliding_window_shapes=(3, size, size),
            strides=(3, stride, stride),
            baselines=BASELINE_VALUE,
        )

        attr_2d = summarize_attribution_map(attr)
        occlusion_maps[model_label][size] = attr_2d

        results.append({
            'model_label': model_label,
            'target_idx': target_idx,
            'occluder_size': size,
            'stride': stride,
            'attr_mean': float(np.mean(attr_2d)),
            'attr_max': float(np.max(attr_2d)),
            'attr_min': float(np.min(attr_2d)),
            'attr_sum_abs': float(np.sum(np.abs(attr_2d))),
        })

results

[{'model_label': 'shape_biased',
  'target_idx': 5,
  'occluder_size': 8,
  'stride': 4,
  'attr_mean': 0.10774524509906769,
  'attr_max': 0.6104068756103516,
  'attr_min': -0.49983859062194824,
  'attr_sum_abs': 7493.87939453125},
 {'model_label': 'shape_biased',
  'target_idx': 5,
  'occluder_size': 16,
  'stride': 8,
  'attr_mean': 0.16731669008731842,
  'attr_max': 1.2846136093139648,
  'attr_min': -0.5214931964874268,
  'attr_sum_abs': 10682.046875},
 {'model_label': 'shape_biased',
  'target_idx': 5,
  'occluder_size': 32,
  'stride': 16,
  'attr_mean': 0.4388270080089569,
  'attr_max': 1.795793056488037,
  'attr_min': -0.3936581611633301,
  'attr_sum_abs': 23954.32421875},
 {'model_label': 'shape_biased',
  'target_idx': 5,
  'occluder_size': 64,
  'stride': 32,
  'attr_mean': 1.1135046482086182,
  'attr_max': 2.7795040607452393,
  'attr_min': -0.3570699691772461,
  'attr_sum_abs': 56921.09765625},
 {'model_label': 'texture_biased',
  'target_idx': 17,
  'occluder_size': 8,
  's

In [1]:
# Cell 8. Show the image
plt.figure(figsize=(5, 5))
plt.imshow(img_np)
plt.title('Input image')
plt.axis('off')
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Cell 9. Visualize occlusion maps for the shape-biased model
n = len(OCCLUDER_SIZES)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for ax, size in zip(axes, OCCLUDER_SIZES):
    heat = occlusion_maps[SHAPE_MODEL_LABEL][size]
    ax.imshow(img_np)
    ax.imshow(normalize_for_display(heat), alpha=0.55)
    ax.set_title(f'{SHAPE_MODEL_LABEL} | size={size}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 10. Visualize occlusion maps for the texture-biased model
n = len(OCCLUDER_SIZES)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
if n == 1:
    axes = [axes]

for ax, size in zip(axes, OCCLUDER_SIZES):
    heat = occlusion_maps[TEXTURE_MODEL_LABEL][size]
    ax.imshow(img_np)
    ax.imshow(normalize_for_display(heat), alpha=0.55)
    ax.set_title(f'{TEXTURE_MODEL_LABEL} | size={size}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11. Plot summary statistics across occluder sizes
import pandas as pd

df = pd.DataFrame(results)
display(df)

plt.figure(figsize=(7, 4))
for model_label, subdf in df.groupby('model_label'):
    subdf = subdf.sort_values('occluder_size')
    plt.plot(subdf['occluder_size'], subdf['attr_sum_abs'], marker='o', label=model_label)

plt.xlabel('Occluder size')
plt.ylabel('Sum of absolute attribution')
plt.title('Occlusion effect summary across sizes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 8. Image list and sweep config
from pathlib import Path

IMAGE_PATHS = [
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/bird_111_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/bird_132_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/bird_136_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/bird_137_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/oyster_4_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/butterfly_144_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/butterfly_11_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/fish_4_symmetrical_2_1/gt.png"),
    Path("/home/hschatzle/monte-carlo-selection/data/cases/sweep/butterfly_560_symmetrical_2_1/gt.png"),
]

# remove duplicates while preserving order
IMAGE_PATHS = list(dict.fromkeys(IMAGE_PATHS))

INPUT_SIZE = 224
OCCLUDER_SIZES = [8, 16, 32]
STRIDE_FRACTION = 0.5
BASELINE_VALUE = 0.0

RESULTS_DIR = Path("/home/hschatzle/monte-carlo-selection/results/occlusion_manual_sweep")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("N images:", len(IMAGE_PATHS))
for p in IMAGE_PATHS:
    print(p)

In [ ]:
# Cell 9. Preprocessing and helper functions
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn.functional as F
from torchvision import transforms

preprocess = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) if USE_IMAGENET_NORMALIZATION else transforms.Lambda(lambda x: x),
])

def load_image_tensor(image_path):
    pil_img = Image.open(image_path).convert("RGB")
    pil_img_resized = pil_img.resize((INPUT_SIZE, INPUT_SIZE))
    img_np_resized = np.asarray(pil_img_resized)
    x = preprocess(pil_img).unsqueeze(0).to(DEVICE)
    return pil_img, pil_img_resized, img_np_resized, x

def get_logits_probs_pred(model, x):
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        pred_idx = int(torch.argmax(logits, dim=1).item())
    return logits, probs, pred_idx

def get_class_name(idx, classes):
    if classes is None:
        return str(idx)
    if idx < 0 or idx >= len(classes):
        return f"idx_{idx}"
    return classes[idx]

def compute_stride(size, fraction=0.5):
    return max(1, int(round(size * fraction)))

def generate_positions(h, w, size, stride):
    ys = list(range(0, max(1, h - size + 1), stride))
    xs = list(range(0, max(1, w - size + 1), stride))

    if ys[-1] != h - size:
        ys.append(h - size)
    if xs[-1] != w - size:
        xs.append(w - size)

    positions = []
    for y0 in ys:
        for x0 in xs:
            positions.append((y0, x0))
    return positions

def apply_occluder(x, y0, x0, size, baseline_value=0.0):
    x_occ = x.clone()
    x_occ[:, :, y0:y0+size, x0:x0+size] = baseline_value
    return x_occ

def normalize_for_display(arr):
    arr = np.asarray(arr, dtype=float)
    a = np.nanmin(arr)
    b = np.nanmax(arr)
    if np.isclose(a, b):
        return np.zeros_like(arr)
    return (arr - a) / (b - a)

In [ ]:
# Cell 10. Manual occlusion sweep
rows = []
baseline_rows = []

for image_path in IMAGE_PATHS:
    pil_img, pil_img_resized, img_np_resized, x = load_image_tensor(image_path)

    for model_label, model in models.items():
        model.eval().to(DEVICE)

        baseline_logits, baseline_probs, baseline_pred_idx = get_logits_probs_pred(model, x)
        baseline_pred_class = get_class_name(baseline_pred_idx, classes)
        baseline_pred_logit = float(baseline_logits[0, baseline_pred_idx].item())
        baseline_pred_prob = float(baseline_probs[0, baseline_pred_idx].item())

        baseline_rows.append({
            "image_path": str(image_path),
            "image_id": image_path.parent.name,
            "model_label": model_label,
            "baseline_pred_idx": baseline_pred_idx,
            "baseline_pred_class": baseline_pred_class,
            "baseline_pred_logit": baseline_pred_logit,
            "baseline_pred_prob": baseline_pred_prob,
        })

        for size in OCCLUDER_SIZES:
            stride = compute_stride(size, STRIDE_FRACTION)
            positions = generate_positions(INPUT_SIZE, INPUT_SIZE, size, stride)

            for (y0, x0) in positions:
                x_occ = apply_occluder(x, y0, x0, size, baseline_value=BASELINE_VALUE)
                occ_logits, occ_probs, occ_pred_idx = get_logits_probs_pred(model, x_occ)

                occ_pred_class = get_class_name(occ_pred_idx, classes)
                occ_pred_logit_for_baseline_class = float(occ_logits[0, baseline_pred_idx].item())
                occ_pred_prob_for_baseline_class = float(occ_probs[0, baseline_pred_idx].item())

                rows.append({
                    "image_path": str(image_path),
                    "image_id": image_path.parent.name,
                    "model_label": model_label,
                    "occluder_size": size,
                    "stride": stride,
                    "y0": y0,
                    "x0": x0,
                    "y_center": y0 + size / 2.0,
                    "x_center": x0 + size / 2.0,
                    "baseline_pred_idx": baseline_pred_idx,
                    "baseline_pred_class": baseline_pred_class,
                    "occluded_pred_idx": occ_pred_idx,
                    "occluded_pred_class": occ_pred_class,
                    "baseline_pred_logit": baseline_pred_logit,
                    "occluded_logit_for_baseline_class": occ_pred_logit_for_baseline_class,
                    "baseline_pred_prob": baseline_pred_prob,
                    "occluded_prob_for_baseline_class": occ_pred_prob_for_baseline_class,
                    "logit_drop": baseline_pred_logit - occ_pred_logit_for_baseline_class,
                    "prob_drop": baseline_pred_prob - occ_pred_prob_for_baseline_class,
                    "prediction_changed": int(occ_pred_idx != baseline_pred_idx),
                    "baseline_class_still_top1": int(occ_pred_idx == baseline_pred_idx),
                    "occluder_area": size * size,
                })

baseline_df = pd.DataFrame(baseline_rows)
position_df = pd.DataFrame(rows)

print("Baseline rows:", len(baseline_df))
print("Position rows:", len(position_df))
display(baseline_df.head())
display(position_df.head())

In [ ]:
# Cell 11. Save raw outputs
baseline_csv = RESULTS_DIR / "baseline_predictions.csv"
positions_csv = RESULTS_DIR / "position_level_occlusion_results.csv"

baseline_df.to_csv(baseline_csv, index=False)
position_df.to_csv(positions_csv, index=False)

print("Saved baseline predictions to:", baseline_csv)
print("Saved position-level results to:", positions_csv)

In [ ]:
# Cell 12. Summary tables
summary_df = (
    position_df
    .groupby(["image_id", "model_label", "occluder_size"], as_index=False)
    .agg(
        n_positions=("x0", "count"),
        mean_logit_drop=("logit_drop", "mean"),
        max_logit_drop=("logit_drop", "max"),
        mean_prob_drop=("prob_drop", "mean"),
        max_prob_drop=("prob_drop", "max"),
        prediction_change_rate=("prediction_changed", "mean"),
        top1_retention_rate=("baseline_class_still_top1", "mean"),
    )
)

summary_df["mean_logit_drop_per_pixel"] = (
    summary_df["mean_logit_drop"] / (summary_df["occluder_size"] ** 2)
)

summary_df["max_logit_drop_per_pixel"] = (
    summary_df["max_logit_drop"] / (summary_df["occluder_size"] ** 2)
)

family_summary_df = (
    summary_df
    .groupby(["model_label", "occluder_size"], as_index=False)
    .agg(
        n_images=("image_id", "nunique"),
        mean_of_mean_logit_drop=("mean_logit_drop", "mean"),
        sd_of_mean_logit_drop=("mean_logit_drop", "std"),
        mean_prediction_change_rate=("prediction_change_rate", "mean"),
        sd_prediction_change_rate=("prediction_change_rate", "std"),
        mean_top1_retention_rate=("top1_retention_rate", "mean"),
        sd_top1_retention_rate=("top1_retention_rate", "std"),
        mean_logit_drop_per_pixel=("mean_logit_drop_per_pixel", "mean"),
        mean_max_logit_drop_per_pixel=("max_logit_drop_per_pixel", "mean"),
    )
)

display(summary_df.head(20))
display(family_summary_df)

In [ ]:
# Cell 13. Save summaries
summary_csv = RESULTS_DIR / "image_level_occlusion_summary.csv"
family_summary_csv = RESULTS_DIR / "family_level_occlusion_summary.csv"

summary_df.to_csv(summary_csv, index=False)
family_summary_df.to_csv(family_summary_csv, index=False)

print("Saved image-level summary to:", summary_csv)
print("Saved family-level summary to:", family_summary_csv)

In [ ]:
# Cell 14. Plot mean logit drop across sizes
plt.figure(figsize=(7, 4))

for model_label, subdf in family_summary_df.groupby("model_label"):
    subdf = subdf.sort_values("occluder_size")
    plt.plot(
        subdf["occluder_size"],
        subdf["mean_of_mean_logit_drop"],
        marker="o",
        label=model_label,
    )

plt.xlabel("Occluder size")
plt.ylabel("Mean logit drop")
plt.title("Mean baseline-class logit drop across occluder sizes")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Cell 15. Plot prediction change rate across sizes
plt.figure(figsize=(7, 4))

for model_label, subdf in family_summary_df.groupby("model_label"):
    subdf = subdf.sort_values("occluder_size")
    plt.plot(
        subdf["occluder_size"],
        subdf["mean_prediction_change_rate"],
        marker="o",
        label=model_label,
    )

plt.xlabel("Occluder size")
plt.ylabel("Prediction change rate")
plt.title("Fraction of occluder positions that change the top-1 prediction")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Cell 16. Plot top-1 retention across sizes
plt.figure(figsize=(7, 4))

for model_label, subdf in family_summary_df.groupby("model_label"):
    subdf = subdf.sort_values("occluder_size")
    plt.plot(
        subdf["occluder_size"],
        subdf["mean_top1_retention_rate"],
        marker="o",
        label=model_label,
    )

plt.xlabel("Occluder size")
plt.ylabel("Top-1 retention rate")
plt.title("Fraction of occluder positions where baseline class stays top-1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Cell 17. Single heatmap view
SELECT_IMAGE_ID = "bird_111_symmetrical_2_1"
SELECT_MODEL_LABEL = "shape_biased"
SELECT_SIZE = 16

image_path = next(p for p in IMAGE_PATHS if p.parent.name == SELECT_IMAGE_ID)
_, pil_img_resized, img_np_resized, _ = load_image_tensor(image_path)

subdf = position_df[
    (position_df["image_id"] == SELECT_IMAGE_ID) &
    (position_df["model_label"] == SELECT_MODEL_LABEL) &
    (position_df["occluder_size"] == SELECT_SIZE)
].copy()

heat = np.full((INPUT_SIZE, INPUT_SIZE), np.nan, dtype=float)
count = np.zeros((INPUT_SIZE, INPUT_SIZE), dtype=float)

for _, r in subdf.iterrows():
    y0 = int(r["y0"])
    x0 = int(r["x0"])
    val = float(r["logit_drop"])
    heat[y0:y0+SELECT_SIZE, x0:x0+SELECT_SIZE] = np.nan_to_num(heat[y0:y0+SELECT_SIZE, x0:x0+SELECT_SIZE], nan=0.0) + val
    count[y0:y0+SELECT_SIZE, x0:x0+SELECT_SIZE] += 1

heat = heat / np.where(count == 0, np.nan, count)

plt.figure(figsize=(6, 6))
plt.imshow(img_np_resized)
plt.imshow(normalize_for_display(heat), alpha=0.55)
plt.title(f"{SELECT_IMAGE_ID} | {SELECT_MODEL_LABEL} | size={SELECT_SIZE}")
plt.axis("off")
plt.show()

In [ ]:
# Cell 18. Side-by-side heatmaps for one image and one size
SELECT_IMAGE_ID = "bird_111_symmetrical_2_1"
SELECT_SIZE = 16

image_path = next(p for p in IMAGE_PATHS if p.parent.name == SELECT_IMAGE_ID)
_, pil_img_resized, img_np_resized, _ = load_image_tensor(image_path)

fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 6))
if len(models) == 1:
    axes = [axes]

for ax, model_label in zip(axes, models.keys()):
    subdf = position_df[
        (position_df["image_id"] == SELECT_IMAGE_ID) &
        (position_df["model_label"] == model_label) &
        (position_df["occluder_size"] == SELECT_SIZE)
    ].copy()

    heat = np.full((INPUT_SIZE, INPUT_SIZE), np.nan, dtype=float)
    count = np.zeros((INPUT_SIZE, INPUT_SIZE), dtype=float)

    for _, r in subdf.iterrows():
        y0 = int(r["y0"])
        x0 = int(r["x0"])
        val = float(r["logit_drop"])
        heat[y0:y0+SELECT_SIZE, x0:x0+SELECT_SIZE] = np.nan_to_num(heat[y0:y0+SELECT_SIZE, x0:x0+SELECT_SIZE], nan=0.0) + val
        count[y0:y0+SELECT_SIZE, x0:x0+SELECT_SIZE] += 1

    heat = heat / np.where(count == 0, np.nan, count)

    ax.imshow(img_np_resized)
    ax.imshow(normalize_for_display(heat), alpha=0.55)
    ax.set_title(f"{model_label} | size={SELECT_SIZE}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 19. Select top-k high-impact positions for stage 2
TOPK = 4
STAGE2_SIZE = 8

topk_df = (
    position_df[position_df["occluder_size"] == STAGE2_SIZE]
    .sort_values(
        ["image_id", "model_label", "logit_drop"],
        ascending=[True, True, False]
    )
    .groupby(["image_id", "model_label"], as_index=False)
    .head(TOPK)
    .copy()
)

display(topk_df[[
    "image_id", "model_label", "occluder_size", "x0", "y0", "logit_drop"
]].sort_values(["image_id", "model_label", "logit_drop"], ascending=[True, True, False]))

In [ ]:
# Cell 20. Stage 2. Multi-occluder targeted masking
def apply_multiple_occluders(x, coords, size, baseline_value=0.0):
    x_occ = x.clone()
    for (y0, x0) in coords:
        x_occ[:, :, y0:y0+size, x0:x0+size] = baseline_value
    return x_occ

stage2_rows = []

for image_path in IMAGE_PATHS:
    image_id = image_path.parent.name
    _, _, _, x = load_image_tensor(image_path)

    for model_label, model in models.items():
        model.eval().to(DEVICE)

        baseline_logits, baseline_probs, baseline_pred_idx = get_logits_probs_pred(model, x)
        baseline_pred_logit = float(baseline_logits[0, baseline_pred_idx].item())
        baseline_pred_prob = float(baseline_probs[0, baseline_pred_idx].item())

        top_sub = topk_df[
            (topk_df["image_id"] == image_id) &
            (topk_df["model_label"] == model_label)
        ].sort_values("logit_drop", ascending=False)

        coords = [(int(r["y0"]), int(r["x0"])) for _, r in top_sub.iterrows()]

        for k in range(1, min(TOPK, len(coords)) + 1):
            chosen = coords[:k]
            x_occ = apply_multiple_occluders(x, chosen, STAGE2_SIZE, baseline_value=BASELINE_VALUE)

            occ_logits, occ_probs, occ_pred_idx = get_logits_probs_pred(model, x_occ)
            occ_pred_logit_for_baseline_class = float(occ_logits[0, baseline_pred_idx].item())
            occ_pred_prob_for_baseline_class = float(occ_probs[0, baseline_pred_idx].item())

            stage2_rows.append({
                "image_id": image_id,
                "image_path": str(image_path),
                "model_label": model_label,
                "occluder_size": STAGE2_SIZE,
                "n_occluders": k,
                "baseline_pred_idx": baseline_pred_idx,
                "occluded_pred_idx": occ_pred_idx,
                "logit_drop": baseline_pred_logit - occ_pred_logit_for_baseline_class,
                "prob_drop": baseline_pred_prob - occ_pred_prob_for_baseline_class,
                "prediction_changed": int(occ_pred_idx != baseline_pred_idx),
                "baseline_class_still_top1": int(occ_pred_idx == baseline_pred_idx),
            })

stage2_df = pd.DataFrame(stage2_rows)
display(stage2_df.head(20))

In [ ]:
# Cell 21. Save and plot stage 2
stage2_csv = RESULTS_DIR / "stage2_multi_occluder_results.csv"
stage2_df.to_csv(stage2_csv, index=False)
print("Saved stage 2 results to:", stage2_csv)

stage2_summary_df = (
    stage2_df
    .groupby(["model_label", "n_occluders"], as_index=False)
    .agg(
        mean_logit_drop=("logit_drop", "mean"),
        sd_logit_drop=("logit_drop", "std"),
        mean_prediction_change_rate=("prediction_changed", "mean"),
        mean_top1_retention_rate=("baseline_class_still_top1", "mean"),
    )
)

display(stage2_summary_df)

plt.figure(figsize=(7, 4))
for model_label, subdf in stage2_summary_df.groupby("model_label"):
    subdf = subdf.sort_values("n_occluders")
    plt.plot(
        subdf["n_occluders"],
        subdf["mean_logit_drop"],
        marker="o",
        label=model_label,
    )

plt.xlabel("Number of targeted occluders")
plt.ylabel("Mean logit drop")
plt.title("Stage 2. Multi-occluder targeted damage")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Cell 22. Optional random-control for stage 2
rng = np.random.default_rng(42)
random_rows = []

for image_path in IMAGE_PATHS:
    image_id = image_path.parent.name
    _, _, _, x = load_image_tensor(image_path)

    for model_label, model in models.items():
        model.eval().to(DEVICE)

        baseline_logits, baseline_probs, baseline_pred_idx = get_logits_probs_pred(model, x)
        baseline_pred_logit = float(baseline_logits[0, baseline_pred_idx].item())
        baseline_pred_prob = float(baseline_probs[0, baseline_pred_idx].item())

        stride = compute_stride(STAGE2_SIZE, STRIDE_FRACTION)
        all_positions = generate_positions(INPUT_SIZE, INPUT_SIZE, STAGE2_SIZE, stride)
        rng.shuffle(all_positions)

        for k in range(1, TOPK + 1):
            chosen = all_positions[:k]
            x_occ = apply_multiple_occluders(x, chosen, STAGE2_SIZE, baseline_value=BASELINE_VALUE)

            occ_logits, occ_probs, occ_pred_idx = get_logits_probs_pred(model, x_occ)
            occ_pred_logit_for_baseline_class = float(occ_logits[0, baseline_pred_idx].item())
            occ_pred_prob_for_baseline_class = float(occ_probs[0, baseline_pred_idx].item())

            random_rows.append({
                "image_id": image_id,
                "image_path": str(image_path),
                "model_label": model_label,
                "occluder_size": STAGE2_SIZE,
                "n_occluders": k,
                "condition": "random",
                "logit_drop": baseline_pred_logit - occ_pred_logit_for_baseline_class,
                "prob_drop": baseline_pred_prob - occ_pred_prob_for_baseline_class,
                "prediction_changed": int(occ_pred_idx != baseline_pred_idx),
                "baseline_class_still_top1": int(occ_pred_idx == baseline_pred_idx),
            })

random_stage2_df = pd.DataFrame(random_rows)
display(random_stage2_df.head())

In [ ]:
# Cell 23. Compare targeted vs random
targeted_for_compare = stage2_df.copy()
targeted_for_compare["condition"] = "targeted"

combined_stage2_df = pd.concat(
    [targeted_for_compare, random_stage2_df],
    ignore_index=True
)

combined_stage2_summary_df = (
    combined_stage2_df
    .groupby(["model_label", "condition", "n_occluders"], as_index=False)
    .agg(
        mean_logit_drop=("logit_drop", "mean"),
        mean_prediction_change_rate=("prediction_changed", "mean"),
        mean_top1_retention_rate=("baseline_class_still_top1", "mean"),
    )
)

display(combined_stage2_summary_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for (model_label, condition), subdf in combined_stage2_summary_df.groupby(["model_label", "condition"]):
    subdf = subdf.sort_values("n_occluders")
    axes[0].plot(
        subdf["n_occluders"],
        subdf["mean_logit_drop"],
        marker="o",
        label=f"{model_label} | {condition}"
    )
    axes[1].plot(
        subdf["n_occluders"],
        subdf["mean_prediction_change_rate"],
        marker="o",
        label=f"{model_label} | {condition}"
    )

axes[0].set_xlabel("Number of occluders")
axes[0].set_ylabel("Mean logit drop")
axes[0].set_title("Targeted vs random. Logit drop")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of occluders")
axes[1].set_ylabel("Prediction change rate")
axes[1].set_title("Targeted vs random. Prediction change")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()